# Spatially-independent validation — *Pinus mugo* 2024

Validation of the 2024 map using **polygon-grouped** train/test splitting as the base method, with a **spatial-block cross-validation sweep** over block sizes of 1–25 km to assess how performance transfers across increasing spatial separation.

Reference polygons are densified to points (NDVI-guided, spatially thinned); features are sampled from both mosaic stacks; all spatial operations use a metric CRS (ETRS89/UTM32N, EPSG:25832). Set `DATA_DIR` / `OUTPUT_DIR` at the top of the config cell.

## 0. Environment

In [ ]:
import importlib, subprocess, sys
for pkg, imp in [("geopandas","geopandas"),("rasterio","rasterio"),
                 ("scikit-learn","sklearn"),("shapely","shapely"),
                 ("scipy","scipy"),("matplotlib","matplotlib"),
                 ("pandas","pandas"),("numpy","numpy")]:
    try: importlib.import_module(imp)
    except ImportError:
        print(f"Installing {pkg} ..."); subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Environment ready. (No skgstat / variogram needed in this version.)")

## 1. Configuration — edit paths and parameters

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio

# --- INPUTS ---
GPKG_PATH   = r"DATA_DIR\Combined\2024\Vectors\pinus_mugo_2024_refpolys_updated20250418.gpkg"
NDVI_RASTER = r"DATA_DIR\Merged_NDVI\merged_2024.tif"          # for clipping/densifying
FEATURE_STACK_MOSAIC1 = r"DATA_DIR\Combined\2024\Rasters\raster_stack_S2_selected30_mosaic1.tif"
FEATURE_STACK_MOSAIC2 = r"DATA_DIR\Combined\2024\Rasters\raster_stack_S2_selected30_mosaic2.tif"
CLASS_COL = "class"

# --- OUTPUTS ---
OUT_DIR = Path(r"DATA_DIR\Combined\2024\SpatialValidation"); OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- POINT DENSIFICATION (mirrors original) ---
NDVI_THRESH_PRESENCE=0.5; NDVI_THRESH_ABSENCE=0.0
THIN_PRESENCE_PIX=3; THIN_ABSENCE_PIX=10

# --- CLEANING (mirrors original cell 11) ---
INVALID_THRESHOLD=-1e20; LOW_NDVI_THRESHOLD=0.1; KEEP_N_FEATURES=30
SEAM_DEDUP_DECIMALS=0          # round coords to nearest metre (UTM/EPSG:25832)

# --- VALIDATION ---
RANDOM_STATE=42; TEST_SIZE=0.30; N_SPATIAL_FOLDS=5
# Metric CRS for distance/blocking. Inputs are EPSG:4326 (degrees); we reproject
# point coordinates to metres for all spatial operations. ETRS89/UTM32N covers
# the German/Austrian Alps. (Pixel SAMPLING still uses each raster's own CRS.)
METRIC_CRS = "EPSG:25832"
# The sweep, in METRES. AOI spans ~250 km, so blocks are coarse: these test
# whether the model transfers across mountain ranges, not just nearby pixels.
BLOCK_SIZES_M = [1000, 2500, 5000, 10000, 25000]
# Buffer defaults to half the block size per sweep entry (set None to disable).
BUFFER_FRACTION = 0.5
print("Config loaded. Block-size sweep:", BLOCK_SIZES_M, "m")

## 2. Reference polygons → NDVI-guided, thinned points (carry `poly_id`)

In [ ]:
import rasterio
from rasterio.mask import mask
from shapely.geometry import Point
from scipy.spatial import cKDTree

polys = gpd.read_file(GPKG_PATH)
if CLASS_COL not in polys.columns:
    raise ValueError(f"GPKG has no '{CLASS_COL}'. Columns: {list(polys.columns)}")
polys = polys.reset_index(drop=True); polys["poly_id"]=polys.index
print(f"{len(polys)} polygons | presence={int((polys[CLASS_COL]==1).sum())} "
      f"absence={int((polys[CLASS_COL]==0).sum())}")

def densify(geom, src, thresh, thin_pix):
    try: out, tr = mask(src, [geom.__geo_interface__], crop=True)
    except Exception: return np.empty((0,2))
    band=out[0]; nod=src.nodata
    r,c=np.where((band>thresh)&(band!=nod)&np.isfinite(band))
    if len(r)==0: return np.empty((0,2))
    xs,ys=rasterio.transform.xy(tr,r,c,offset="center")
    co=np.column_stack([np.asarray(xs),np.asarray(ys)])
    mind=thin_pix*src.res[0]; tree=cKDTree(co); keep=np.ones(len(co),bool); sel=[]
    for i in range(len(co)):
        if keep[i]:
            sel.append(i); keep[tree.query_ball_point(co[i],mind)]=False; keep[i]=True
    return co[sel]

recs=[]
with rasterio.open(NDVI_RASTER) as src:
    pr=polys.to_crs(src.crs)
    for _,row in pr.iterrows():
        cls=int(row[CLASS_COL])
        thr=NDVI_THRESH_PRESENCE if cls==1 else NDVI_THRESH_ABSENCE
        thin=THIN_PRESENCE_PIX if cls==1 else THIN_ABSENCE_PIX
        for x,y in densify(row.geometry,src,thr,thin):
            recs.append((x,y,int(row["poly_id"]),cls))
    pts_crs=src.crs
pts=gpd.GeoDataFrame(pd.DataFrame(recs,columns=["x","y","poly_id","type"]),
                     geometry=[Point(x,y) for x,y,_,_ in recs], crs=pts_crs)
print(f"{len(pts)} points from {pts.poly_id.nunique()} polygons")

## 3. Sample features from both mosaics; concatenate with identities

In [ ]:
def extract(points_gdf, stack_path):
    out=[]
    with rasterio.open(stack_path) as src:
        g_native=points_gdf.to_crs(src.crs)        # for pixel sampling
        g_metric=points_gdf.to_crs(METRIC_CRS)     # for x/y in metres
        nbnd=src.count
        names=[src.descriptions[i] if src.descriptions[i] else f"Band_{i+1}" for i in range(nbnd)]
        arr=src.read()
        for idx,gn,gm in zip(g_native.index,g_native.geometry,g_metric.geometry):
            try: r,c=src.index(gn.x,gn.y)
            except Exception: continue
            if 0<=r<src.height and 0<=c<src.width:
                out.append([points_gdf.at[idx,"poly_id"],points_gdf.at[idx,"type"],
                            gm.x,gm.y]+list(arr[:,r,c].astype(float)))
    return pd.DataFrame(out,columns=["poly_id","type","x","y"]+names),names

df1,n1=extract(pts,FEATURE_STACK_MOSAIC1)
df2,n2=extract(pts,FEATURE_STACK_MOSAIC2)
print(f"Mosaic1 {len(df1)} rows | Mosaic2 {len(df2)} rows")
band_names=[c for c in n1 if c in n2]
keep=["poly_id","type","x","y"]+band_names
feat=pd.concat([df1[keep],df2[keep]],ignore_index=True)
feat["point_uid"]=(feat.x.round(SEAM_DEDUP_DECIMALS).astype(str)+"_"+
                   feat.y.round(SEAM_DEDUP_DECIMALS).astype(str))
print(f"Concatenated {len(feat)} rows | seam-duplicate rows: "
      f"{int(feat.point_uid.duplicated(keep=False).sum())}")
# SANITY CHECK — these must be metres (UTM), not degrees:
print(f"Point x range (m): {feat.x.min():,.0f} to {feat.x.max():,.0f}")
print(f"Point y range (m): {feat.y.min():,.0f} to {feat.y.max():,.0f}")
assert feat.x.max()-feat.x.min() > 1000, 'Coordinates look like degrees, not metres — check METRIC_CRS!'

In [ ]:
# Original cleaning, applied once before any split.
for c in band_names:
    feat[c]=feat[c].where(feat[c]>INVALID_THRESHOLD,np.nan)
n0=len(feat)
slope=next((c for c in band_names if "slope" in c.lower()),None)
aspect=next((c for c in band_names if "aspect" in c.lower()),None)
if slope and aspect:
    feat=feat[feat[slope].between(0,90)&feat[aspect].between(0,360)]
    print(f"Removed {n0-len(feat)} rows invalid slope/aspect")
ndvi_col=next((c for c in band_names if "NDVI" in c.upper()),None)
if ndvi_col:
    feat[ndvi_col]=pd.to_numeric(feat[ndvi_col],errors="coerce")
    drop=(feat["type"]==1)&(feat[ndvi_col]<LOW_NDVI_THRESHOLD)
    print(f"Dropping {int(drop.sum())} presence points NDVI<{LOW_NDVI_THRESHOLD}")
    feat=feat[~drop]
else:
    print("⚠️ No NDVI column — low-NDVI filter skipped")
FEATURES=band_names[:KEEP_N_FEATURES]
if ndvi_col and ndvi_col not in FEATURES: FEATURES=FEATURES[:KEEP_N_FEATURES-1]+[ndvi_col]
feat=feat.reset_index(drop=True)
print(f"{len(feat)} points | {len(FEATURES)} features | classes {feat.type.value_counts().to_dict()}")

## 4. Evaluator (fold-internal imputation, per-class metrics)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score,f1_score,precision_score,
                             recall_score,cohen_kappa_score)
from sklearn.model_selection import GroupKFold

def make_rf(): return RandomForestClassifier(n_estimators=100,random_state=RANDOM_STATE,n_jobs=-1)
def evaluate(tr,te,label):
    m=tr[FEATURES].mean(numeric_only=True)
    clf=make_rf().fit(tr[FEATURES].fillna(m),tr["type"])
    p=clf.predict(te[FEATURES].fillna(m)); y=te["type"]
    return {"scheme":label,"n_train":len(tr),"n_test":len(te),
            "accuracy":accuracy_score(y,p),"kappa":cohen_kappa_score(y,p),
            "f1_presence":f1_score(y,p,pos_label=1,zero_division=0),
            "prec_presence":precision_score(y,p,pos_label=1,zero_division=0),
            "rec_presence":recall_score(y,p,pos_label=1,zero_division=0),
            "f1_absence":f1_score(y,p,pos_label=0,zero_division=0)}

## 5. Base method — polygon-grouped split
Points from the same reference polygon are kept entirely within either train or test, so the test set is spatially independent of training at the polygon level. This is the base validation reported for the map.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

i, j = next(GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
            .split(feat, feat["type"], groups=feat["poly_id"]))
res_polygon = evaluate(feat.iloc[i], feat.iloc[j], "Polygon-grouped")
print("Polygon-grouped split:")
for k, v in res_polygon.items():
    print(f"  {k}: {round(v,3) if isinstance(v,float) else v}")


## 6. Spatial-block CV sweep (1–25 km)
Square spatial blocks of increasing size define the CV folds; an optional buffer (half the block size) removes training points near each test block. Coarser blocks test transfer across greater spatial separation across the ~250 km study extent.

In [ ]:
def block_cv(block_m, buffer_m):
    X0,Y0=feat.x.min(),feat.y.min()
    bid=(((feat.x-X0)//block_m).astype(int).astype(str)+"_"+
         ((feat.y-Y0)//block_m).astype(int).astype(str)).values
    nb_=len(set(bid))
    n_splits=min(N_SPATIAL_FOLDS,nb_)
    if n_splits<2:
        print(f"  block {block_m} m: only {nb_} block(s) — skipped"); return None
    gkf=GroupKFold(n_splits=n_splits); fm=[]
    for ti,vi in gkf.split(feat,feat["type"],groups=bid):
        te=feat.iloc[vi]; tr=feat.iloc[ti]
        if buffer_m and buffer_m>0:
            d,_=cKDTree(te[["x","y"]].values).query(tr[["x","y"]].values,k=1)
            tr=tr[d>buffer_m]
        if tr["type"].nunique()<2 or te["type"].nunique()<2: continue
        fm.append(evaluate(tr,te,f"blk{block_m}"))
    if not fm:
        print(f"  block {block_m} m: no usable folds"); return None
    d=pd.DataFrame(fm)
    return {"block_m":block_m,"buffer_m":buffer_m,"n_blocks":nb_,"n_folds":len(d),
            "f1_presence":d.f1_presence.mean(),"f1_presence_std":d.f1_presence.std(),
            "accuracy":d.accuracy.mean(),"kappa":d.kappa.mean(),
            "prec_presence":d.prec_presence.mean(),"rec_presence":d.rec_presence.mean()}

sweep=[]
for b in BLOCK_SIZES_M:
    buf=(BUFFER_FRACTION*b) if BUFFER_FRACTION else 0.0
    r=block_cv(b,buf)
    if r: sweep.append(r); print(f"  block {b} m: F1_presence={r['f1_presence']:.3f} "
                                 f"(±{r['f1_presence_std']:.3f}, {r['n_folds']} folds)")
sweep_df=pd.DataFrame(sweep)
sweep_df.round(3) if len(sweep_df) else print("⚠️ No block size produced usable folds.")

## 7. Summary — polygon split and block-size sweep

In [ ]:
summary = pd.DataFrame([res_polygon] +
    [{"scheme": f"block {r['block_m']/1000:.0f} km",
      "n_test": r["n_folds"], "accuracy": r["accuracy"], "kappa": r["kappa"],
      "f1_presence": r["f1_presence"], "prec_presence": r["prec_presence"],
      "rec_presence": r["rec_presence"]} for r in sweep])
summary.round(3)


In [ ]:
# Sweep plot for the supplement.
if len(sweep_df):
    import matplotlib.pyplot as plt
    fig,ax=plt.subplots(figsize=(7,4.5))
    ax.errorbar(sweep_df.block_m,sweep_df.f1_presence,yerr=sweep_df.f1_presence_std,
                marker="o",capsize=4,label="Block CV (presence F1)")
    ax.axhline(res_A1["f1_presence"],ls="--",color="grey",label="A1 random (raw)")
    ax.axhline(res_A2["f1_presence"],ls=":",color="tab:red",label="A2 seam-dedup")
    ax.axhline(res_B["f1_presence"],ls="-.",color="tab:green",label="B polygon-grouped")
    ax.set_xlabel("Block size (m)"); ax.set_ylabel("Presence-class F1")
    ax.set_title("Validation F1 vs spatial separation"); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(OUT_DIR/"block_size_sweep.png",dpi=150)
    print("Saved block_size_sweep.png")

## 8. Reading the result
The polygon-grouped split gives the headline validation metrics for the 2024 map. The block-size sweep shows how presence-class F1 changes as spatial separation between training and test data increases from 1 to 25 km, characterising the spatial transferability of the classifier rather than near-pixel agreement.